# Aula 08 — Segmentação e quantificação de microscopia

**Processamento de Linguagem Natural e Imagens · 2026.2 · Prof. James Moraes de Almeida**

*Segmentar* é decidir, para cada pixel, a que objeto ele pertence. Nesta aula vamos:

1. separar objeto e fundo por **limiar**, à mão e pelo **método de Otsu**;
2. separar **mais de duas fases** (multi-Otsu) e regiões por **textura**;
3. **medir** cada objeto: área, diâmetro, em unidades físicas;
4. separar objetos **encostados** com o **watershed**;
5. achar objetos por **forma** (Hough) e por **modelo** (template matching);
6. responder o enigma da aula: **qual o diâmetro das nanopartículas de sílica** que
   vocês sintetizaram e fotografaram no LNNano?

> O código parte dos tutoriais 51 a 58 do curso *Python for Microscopists*, de
> Sreenivas Bhattiprolu (canal DigitalSreeni):
> https://github.com/bnsreenu/python_for_image_processing_APEER

## Antes de começar

Use o mesmo ambiente da Aula 07:

```bash
conda activate img_proc
```

As imagens estão na pasta `images/`, ao lado deste notebook. Nada é gravado de volta
em `images/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage as ndi
from skimage import io, filters, measure, morphology, segmentation, feature
from skimage.color import label2rgb, rgb2gray
from skimage.util import img_as_ubyte

IMAGENS = Path("images")
plt.rcParams["figure.dpi"] = 110


def mostra(*imagens, titulos=None, cmap="gray", altura=3.5):
    """Mostra várias imagens lado a lado, sem eixos."""
    n = len(imagens)
    fig, eixos = plt.subplots(1, n, figsize=(altura * 1.35 * n, altura))
    eixos = np.atleast_1d(eixos)
    for k, (ax, im) in enumerate(zip(eixos, imagens)):
        ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
        ax.set_axis_off()
        if titulos:
            ax.set_title(titulos[k])
    plt.tight_layout()
    plt.show()

# 1 — Limiar: à mão e pelo método de Otsu

*(tutorial 51)*

Células de osteossarcoma em fluorescência. Os núcleos estão marcados com DAPI, que
brilha no **canal azul**.

> O `io.imread` do scikit-image devolve os canais na ordem **RGB**, então o azul é o
> índice 2. (O `cv2.imread` devolve **BGR**, e o azul seria o índice 0 — o tutorial
> original usava esse.)

In [ ]:
celulas = io.imread(IMAGENS / "Osteosarcoma_01.tif")
azul = celulas[..., 2]
print("formato:", celulas.shape, "· tipo:", celulas.dtype)
mostra(celulas, azul, titulos=["as três cores", "canal azul (núcleos)"])

O histograma mostra duas populações: o fundo escuro, com muitos pixels, e os núcleos.

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(azul.ravel(), bins=100, range=(0, 150), color="steelblue")
plt.xlabel("nível de cinza"); plt.ylabel("número de pixels")
plt.show()

## 1.1 — Limiar escolhido à mão

Olhando o histograma, 40 parece separar as duas populações.

In [ ]:
nucleos_mao = azul > 40
mostra(azul, nucleos_mao, titulos=["canal azul", "acima de 40"])

## 1.2 — O método de Otsu

Otsu (1979) escolhe o limiar que **mais separa** as duas classes. Para cada limiar
possível $t$, os pixels se dividem em duas classes, com pesos $w_1, w_2$ (a fração de
pixels em cada uma) e médias $\mu_1, \mu_2$. A separação é a **variância entre
classes**:

$$\sigma_b^2(t) = w_1(t)\,w_2(t)\,\big[\mu_1(t) - \mu_2(t)\big]^2$$

com

$$w_1(t) = \sum_{i=0}^{t} P(i), \qquad w_2(t) = 1 - w_1(t), \qquad
\mu_1(t) = \frac{1}{w_1}\sum_{i=0}^{t} i\,P(i), \qquad
\mu_2(t) = \frac{1}{w_2}\sum_{i=t+1}^{255} i\,P(i)$$

onde $P(i)$ é a fração dos pixels com nível $i$. O limiar de Otsu é o $t$ que maximiza
$\sigma_b^2$.

> Os slides de 2025.2 dividiam $\mu_2$ por $w_1$ e esqueciam o quadrado. Estas são as
> equações certas — e a célula abaixo as confere contra a biblioteca.

In [ ]:
def otsu_a_mao(imagem):
    """Varre todos os limiares de uma imagem uint8 e devolve (t, curva σ²_b)."""
    n = np.bincount(imagem.ravel(), minlength=256).astype(float)
    P = n / n.sum()
    i = np.arange(256)
    w1 = np.cumsum(P)
    w2 = 1 - w1
    soma1 = np.cumsum(i * P)
    with np.errstate(divide="ignore", invalid="ignore"):
        mu1 = soma1 / w1
        mu2 = (soma1[-1] - soma1) / w2
        sb = np.nan_to_num(w1 * w2 * (mu1 - mu2) ** 2)
    return int(np.argmax(sb)), sb


t_mao, curva = otsu_a_mao(azul)
t_skimage = filters.threshold_otsu(azul)
t_cv2, _ = cv2.threshold(azul, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu à mão:", t_mao, "· scikit-image:", t_skimage, "· OpenCV:", int(t_cv2))

plt.figure(figsize=(6, 3))
plt.plot(curva, color="seagreen")
plt.axvline(t_mao, color="firebrick", ls="--")
plt.xlabel("limiar t"); plt.ylabel("σ²_b  (níveis de cinza)²")
plt.xlim(0, 150)
plt.show()

As três dão o mesmo número. Maximizar a variância **entre** as classes é o mesmo que
minimizar a variância **dentro** de cada uma, porque a soma das duas é a variância
total da imagem, que não depende de $t$.

In [ ]:
nucleos = azul > t_mao
mostra(nucleos_mao, nucleos, titulos=["à mão (40)", f"Otsu ({t_mao})"])

# 2 — Mais de duas fases: multi-Otsu

*(tutorial 52)*

Uma imagem de microscopia eletrônica com elétrons retroespalhados (BSE): o nível de
cinza depende da composição, e há quatro fases. O multi-Otsu generaliza a ideia:
procura **vários** limiares que maximizam a separação entre as classes.

In [ ]:
bse = io.imread(IMAGENS / "BSE.tif")
if bse.ndim == 3:
    bse = bse[..., 0]          # a imagem vem com três canais iguais
print("formato:", bse.shape)

plt.figure(figsize=(6, 3))
plt.hist(bse.ravel(), bins=128, range=(0, 255), color="gray")
plt.xlabel("nível de cinza"); plt.ylabel("número de pixels")
plt.show()

In [ ]:
limiares = filters.threshold_multiotsu(bse, classes=4)
fases = np.digitize(bse, bins=limiares)      # 0, 1, 2, 3 por pixel
print("limiares:", limiares)
mostra(bse, fases, titulos=["BSE", "quatro fases"], cmap="viridis")

## 2.1 — Limpando com morfologia

Pixels isolados e buraquinhos sujam as fases. A **abertura** (erosão seguida de
dilatação) apaga ilhas menores que o elemento estruturante; o **fechamento** (o
contrário) tapa buracos pequenos. Uma fase de cada vez:

In [ ]:
quadrado = np.ones((3, 3), bool)
limpas = np.zeros_like(fases)
for k in range(4):
    fase = fases == k
    fase = ndi.binary_closing(ndi.binary_opening(fase, quadrado), quadrado)
    limpas[fase] = k
mostra(fases, limpas, titulos=["antes", "depois da abertura e do fechamento"], cmap="viridis")

In [ ]:
# fração de área de cada fase — o que um metalurgista de fato quer saber
for k in range(4):
    print(f"fase {k}: {100 * (limpas == k).mean():.1f} % da área")

# 3 — Textura: o ensaio de cicatrização (*scratch assay*)

*(tutoriais 53 e 54)*

Risca-se uma camada de células e acompanha-se o fechamento do risco ao longo do tempo.
O risco **não é mais escuro** que as células: é mais **liso**. O limiar no nível de
cinza não funciona; é preciso medir **textura**.

In [ ]:
risco = io.imread(IMAGENS / "scratch0.jpg", as_gray=True)
print("tipo:", risco.dtype, "· de", risco.min(), "a", risco.max())
mostra(risco, titulos=["ensaio de cicatrização, t = 0"])

## 3.1 — Variância local

A variância numa janela 7×7 é alta onde há células (textura) e baixa no risco.
Funciona, mas é ruidosa.

In [ ]:
k = 7
media = ndi.uniform_filter(risco, k)
media_quadrado = ndi.uniform_filter(risco ** 2, k)
variancia = media_quadrado - media ** 2
mostra(risco, variancia, titulos=["imagem", "variância local 7×7"])

## 3.2 — Entropia local

A **entropia** mede a desordem dos níveis de cinza numa vizinhança. Região de células:
alta; risco: baixa. Depois, o Otsu separa as duas.

> O filtro de Gabor, que o tutorial também testa aqui, fica para a Aula 09, onde ele
> vira *feature* de aprendizado de máquina.

In [ ]:
from skimage.filters.rank import entropy

risco8 = img_as_ubyte(risco)
ent = entropy(risco8, morphology.disk(3))
t_ent = filters.threshold_otsu(ent)
area_risco = ent <= t_ent
mostra(ent, area_risco, titulos=["entropia local", f"entropia ≤ {t_ent:.2f} (Otsu)"])

In [ ]:
ESCALA_RISCO = 0.45      # µm por pixel, valor do tutorial original
n_px = area_risco.sum()
print(f"área do risco: {n_px} pixels = {n_px * ESCALA_RISCO ** 2:.0f} µm²")

## 3.3 — A série temporal

O mesmo procedimento em cada imagem da pasta, em ordem, e uma reta pelos pontos: a
inclinação é a **velocidade de fechamento**.

In [ ]:
from scipy.stats import linregress

arquivos = sorted((IMAGENS / "scratch_assay").glob("scratch*.jpg"),
                  key=lambda p: int(p.stem.removeprefix("scratch")))
horas, areas = [], []
for h, arq in enumerate(arquivos):          # uma imagem por hora
    im = img_as_ubyte(io.imread(arq, as_gray=True))
    e = entropy(im, morphology.disk(3))
    areas.append((e <= filters.threshold_otsu(e)).sum() * ESCALA_RISCO ** 2)
    horas.append(h)

ajuste = linregress(horas, areas)
plt.figure(figsize=(5, 3))
plt.plot(horas, areas, "o", color="steelblue")
plt.plot(horas, ajuste.intercept + ajuste.slope * np.array(horas), color="firebrick")
plt.xlabel("tempo (h)"); plt.ylabel("área do risco (µm²)")
plt.show()
print(f"fechamento: {-ajuste.slope:.0f} µm²/h · R² = {ajuste.rvalue ** 2:.3f}")

# 4 — Medindo cada objeto: `regionprops`

*(tutorial 55)*

Ferro fundido: as manchas escuras são grafita. Depois do limiar, `measure.label`
numera cada região conectada, e `regionprops` mede cada uma.

In [ ]:
ferro = img_as_ubyte(rgb2gray(io.imread(IMAGENS / "cast_iron1.tif")))
t_ferro = filters.threshold_otsu(ferro)
grafita = ferro < t_ferro                       # a grafita é a parte ESCURA
grafita = segmentation.clear_border(grafita)    # objeto cortado pela borda mede errado
rotulos = measure.label(grafita)
print("limiar:", t_ferro, "· regiões:", rotulos.max())
mostra(ferro, label2rgb(rotulos, image=ferro, bg_label=0),
       titulos=["ferro fundido", "cada região, uma cor"])

In [ ]:
ESCALA_FERRO = 0.6       # µm por pixel, valor do tutorial original
tabela = pd.DataFrame(measure.regionprops_table(
    rotulos, ferro,
    properties=["label", "area", "equivalent_diameter_area", "intensity_mean", "solidity"]))
tabela = tabela[tabela["area"] > 50]            # descarta ruído
tabela["area_um2"] = tabela["area"] * ESCALA_FERRO ** 2
tabela["diametro_um"] = tabela["equivalent_diameter_area"] * ESCALA_FERRO
tabela.head()

In [ ]:
tabela["diametro_um"].plot.hist(bins=15, color="steelblue", figsize=(5, 3))
plt.xlabel("diâmetro equivalente (µm)"); plt.ylabel("número de nódulos")
plt.show()

> **Diâmetro equivalente** é o diâmetro do círculo de mesma área. Para objetos
> redondos, é o diâmetro; para objetos alongados, é só um número de tamanho.
> A **solidez** (área ÷ área do envoltório convexo) diz o quanto o objeto é "cheio":
> perto de 1, convexo; bem abaixo de 1, recortado — ou dois objetos grudados.

# 5 — Detector de manchas (*blobs*)

*(tutorial 56)*

O `SimpleBlobDetector` do OpenCV faz tudo de uma vez: limiariza em vários níveis, agrupa
pixels conectados e filtra por tamanho e forma.

In [ ]:
param = cv2.SimpleBlobDetector_Params()
param.minThreshold, param.maxThreshold = 0, 255
param.filterByArea, param.minArea, param.maxArea = True, 50, 10000
param.filterByColor, param.blobColor = True, 0       # manchas escuras
param.filterByCircularity, param.minCircularity = True, 0.3
param.filterByConvexity = False        # o OpenCV 5 exige minConvexity > 0 quando ligado
param.filterByInertia, param.minInertiaRatio = True, 0.5
detector = cv2.SimpleBlobDetector_create(param)
manchas = detector.detect(ferro)
print("manchas encontradas:", len(manchas))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.imshow(ferro, cmap="gray")
for m in manchas:
    ax.add_patch(plt.Circle(m.pt, m.size / 2, fill=False, color="red", lw=1))
ax.set_axis_off()
plt.show()

# 6 — Objetos encostados: o watershed

*(tutorial 57)*

Núcleos que se tocam viram **uma** região depois do limiar. O watershed trata a imagem
como um relevo e "enche de água" a partir de marcadores, um por objeto; onde duas águas
se encontram, levanta uma linha divisória.

O roteiro clássico do OpenCV:

1. limiar e abertura → a máscara;
2. dilatar a máscara → o que é **com certeza fundo**;
3. distância até o fundo, limiarizada → o que é **com certeza núcleo** (os centros);
4. o que sobra entre os dois é **desconhecido** — o watershed decide.

In [ ]:
ESCALA_NUCLEO = 0.454    # µm por pixel, dos metadados da imagem original
_, masc = cv2.threshold(azul, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
k3 = np.ones((3, 3), np.uint8)
masc = cv2.morphologyEx(masc, cv2.MORPH_OPEN, k3, iterations=2)
masc = segmentation.clear_border(masc)

fundo_certo = cv2.dilate(masc, k3, iterations=10)
dist = cv2.distanceTransform(masc, cv2.DIST_L2, 5)
_, centros = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
centros = centros.astype(np.uint8)
desconhecido = cv2.subtract(fundo_certo, centros)
mostra(masc, dist, centros, desconhecido,
       titulos=["máscara", "distância até o fundo", "com certeza núcleo", "desconhecido"],
       altura=2.8)

In [ ]:
_, marcadores = cv2.connectedComponents(centros)
marcadores = marcadores + 10           # o fundo certo não pode ser 0
marcadores[desconhecido == 255] = 0    # 0 = o watershed decide
bgr = cv2.cvtColor(celulas, cv2.COLOR_RGB2BGR)   # o cv2.watershed quer 3 canais BGR
marcadores = cv2.watershed(bgr, marcadores)      # fronteiras recebem -1

contorno = celulas.copy()
contorno[marcadores == -1] = [255, 255, 0]
mostra(contorno, label2rgb(marcadores, bg_label=10),
       titulos=["fronteiras do watershed", "cada núcleo, uma cor"])

In [ ]:
nuc = pd.DataFrame(measure.regionprops_table(
    np.where(marcadores > 10, marcadores, 0), azul,
    properties=["label", "area", "equivalent_diameter_area", "solidity"]))
nuc = nuc[nuc["area"] > 50]
nuc["diametro_um"] = nuc["equivalent_diameter_area"] * ESCALA_NUCLEO
print(f"{len(nuc)} núcleos · diâmetro médio {nuc['diametro_um'].mean():.1f} µm")

# 7 — Achar por modelo: *template matching*

*(tutorial 58)*

Desliza-se um recorte (o modelo) sobre a imagem e mede-se, em cada posição, o quanto
ele se parece com a vizinhança — a mesma janela deslizante do kernel da Aula 07.

In [ ]:
po = io.imread(IMAGENS / "Ti_powder.tif")
po = img_as_ubyte(rgb2gray(po)) if po.ndim == 3 else po
modelo = io.imread(IMAGENS / "Ti_powder_single.tif")
modelo = img_as_ubyte(rgb2gray(modelo)) if modelo.ndim == 3 else modelo
h, w = modelo.shape

semelhanca = cv2.matchTemplate(po, modelo, cv2.TM_CCOEFF_NORMED)
_, melhor, _, onde = cv2.minMaxLoc(semelhanca)
print(f"melhor correspondência: {melhor:.3f} em {onde}")
mostra(modelo, semelhanca, titulos=["o modelo", "semelhança em cada posição"])

In [ ]:
LIMIAR_SEM = 0.8
picos = feature.peak_local_max(semelhanca, min_distance=min(h, w) // 2,
                               threshold_abs=LIMIAR_SEM)
brutos = (semelhanca >= LIMIAR_SEM).sum()
print(f"posições acima de {LIMIAR_SEM}: {brutos} · partículas (picos locais): {len(picos)}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(po, cmap="gray")
for y, x in picos:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="red", lw=1))
ax.set_axis_off()
plt.show()

> O tutorial original desenhava um retângulo para **cada posição** acima do limiar —
> várias por partícula. Ficar só com os **picos locais** dá uma caixa por partícula.

# 8 — A prática: o diâmetro das nanopartículas de sílica

A micrografia que vocês fizeram no LNNano (MEV, 10 000×). A pergunta do início da
aula: **qual o diâmetro destas partículas?**

In [ ]:
mev = io.imread(IMAGENS / "SiO2-MEV.jpg")
print("formato:", mev.shape, "· tipo:", mev.dtype)
mostra(mev, titulos=["SiO₂, MEV"], altura=5)

## 8.1 — A escala, medida na própria imagem

A tarja preta do microscópio traz uma barra de **2 µm**. Primeiro, onde começa a tarja
(a primeira linha com mais da metade dos pixels quase pretos); depois, a barra: a linha
clara mais longa dentro dela, de um traço vertical ao outro.

In [ ]:
tarja = min(y for y in range(mev.shape[0]) if (mev[y] < 20).mean() > 0.5)

melhor = (0, None)
for y in range(tarja, mev.shape[0]):
    claro = (mev[y] > 150).astype(int)
    d = np.diff(np.r_[0, claro, 0])
    ini, fim = np.where(d == 1)[0], np.where(d == -1)[0]
    if len(ini) and (fim - ini).max() > melhor[0]:
        k = np.argmax(fim - ini)
        melhor = (fim[k] - ini[k], (y, ini[k], fim[k] - 1))
y_barra, xe, xd = melhor[1]

# os dois traços verticais: colunas claras logo acima da linha da barra
acima = mev[y_barra - 5:y_barra, xe - 2:xd + 3] > 150
cols = np.where(acima.all(axis=0))[0] + xe - 2
meio = (xe + xd) / 2
barra_px = cols[cols > meio].mean() - cols[cols < meio].mean()
NM_POR_PX = 2000 / barra_px
print(f"tarja a partir da linha {tarja}")
print(f"barra: {barra_px:.1f} px = 2 µm  →  1 px = {NM_POR_PX:.2f} nm")

In [ ]:
foto = mev[:tarja - 3]          # a micrografia sem a tarja
mostra(foto, mev[tarja - 2:, int(xe) - 30:int(xd) + 120], titulos=["sem a tarja", "a barra"])

## 8.2 — A intuição: limiar e contar manchas

Suaviza, limiariza pelo Otsu, limpa com uma abertura e conta as manchas.

In [ ]:
suave = np.round(filters.gaussian(foto.astype(float), 2, preserve_range=True)).astype(np.uint8)
t_mev = filters.threshold_otsu(suave)
mascara = morphology.opening(suave > t_mev, morphology.disk(2))

manchas = measure.label(mascara)
inteiras = segmentation.clear_border(manchas)
d_manchas = np.array([p.equivalent_diameter_area for p in measure.regionprops(inteiras)])
print(f"Otsu: t = {t_mev}")
print(f"{manchas.max()} manchas ({len(d_manchas)} sem tocar a borda)")
print(f"diâmetro médio das manchas inteiras: {d_manchas.mean() * NM_POR_PX:.0f} nm")
mostra(mascara, label2rgb(manchas, bg_label=0), titulos=["máscara", "cada mancha, uma cor"])

**Não pode estar certo.** Olhando a foto, as partículas têm uns 35 pixels de diâmetro,
cerca de 600 nm. As manchas são **aglomerados**: partículas encostadas viram uma só.

### Um aviso sobre `binary_fill_holes`

Muitos tutoriais preenchem os buracos da máscara depois do limiar. Aqui isso é um
desastre: o espaço escuro cercado por um anel de partículas **é** um buraco, e o
preenchimento funde o anel inteiro numa flor.

In [ ]:
cheia = ndi.binary_fill_holes(mascara)
print("pixels de fundo pintados de branco pelo preenchimento:", int((cheia & ~mascara).sum()))
mostra(mascara, cheia, titulos=["máscara", "com os buracos preenchidos"])

## 8.3 — O watershed

O roteiro do scikit-image, mais direto que o do OpenCV da seção 6:

1. a **distância** de cada pixel até o fundo (suavizada, para cada partícula ter um
   pico só);
2. os **marcadores**: os picos locais da distância, a pelo menos 10 px um do outro;
3. o watershed sobre a distância **invertida** (os picos viram vales).

In [ ]:
dist = ndi.distance_transform_edt(mascara)
relevo = filters.gaussian(dist, 1)
picos = feature.peak_local_max(relevo, min_distance=10, labels=measure.label(mascara),
                               exclude_border=False)
marc = np.zeros(mascara.shape, int)
marc[tuple(picos.T)] = np.arange(1, len(picos) + 1)
bacias = segmentation.watershed(-relevo, marc, mask=mascara)
print("marcadores:", len(picos))
mostra(relevo, label2rgb(bacias, foto, bg_label=0, alpha=0.4),
       titulos=["distância até o fundo", "as bacias"])

Agora a medida. Ficam de fora as partículas cortadas pela borda e as regiões com
solidez abaixo de 0,9 (pedaços de partícula, ou o substrato exposto no canto inferior
direito).

In [ ]:
props = measure.regionprops(segmentation.clear_border(bacias))
d_ws = np.array([p.equivalent_diameter_area for p in props if p.solidity > 0.9])
print(f"{len(props)} partículas inteiras, {len(d_ws)} com solidez > 0,9")
print(f"diâmetro médio: {d_ws.mean() * NM_POR_PX:.0f} ± {d_ws.std() * NM_POR_PX:.0f} nm")

plt.figure(figsize=(5, 3))
plt.hist(d_ws * NM_POR_PX, bins=np.arange(250, 950, 50), color="steelblue")
plt.xlabel("diâmetro equivalente (nm)"); plt.ylabel("partículas")
plt.show()

## 8.4 — Outra rota: a transformada de Hough

O notebook original desta prática procurava **círculos** com `cv2.HoughCircles`. O
Hough não usa limiar de intensidade nem máscara (os limiares dele são no gradiente e
nos votos): cada pixel de borda "vota" nos centros de círculo
compatíveis com ele, e os centros mais votados ganham. É uma rota **independente** do
watershed — se as duas concordarem, a medida é confiável.

Os parâmetros são os do notebook original:

- `dp=2` — o acumulador de votos tem metade da resolução da imagem;
- `minDist=30` — distância mínima entre centros, em pixels;
- `param1=100`, `param2=35` — os limiares do detector de bordas e dos votos;
- `minRadius=12`, `maxRadius=22` — a faixa de raios procurada, em pixels.

Antes, um filtro de **mediana** 9×9 tira o ruído. (O comentário do original dizia
"filtro Gaussiano com desvio padrão de 9", mas o código é um `medianBlur`.)

In [ ]:
HOUGH = dict(dp=2.0, minDist=30, param1=100, param2=35, minRadius=12, maxRadius=22)

def circulos(imagem):
    return cv2.HoughCircles(cv2.medianBlur(imagem, 9), cv2.HOUGH_GRADIENT, **HOUGH)[0]

c_toda = circulos(mev)

# A caixa preta de texto: o maior pedaço de pixels escuros nas linhas da tarja. Ela NÃO
# ocupa a largura toda: ao lado dela e embaixo dela há partículas de verdade.
escuro = mev < 20
escuro[:tarja] = False
rot_caixa = measure.label(escuro)
caixa = rot_caixa == np.argmax(np.bincount(rot_caixa.ravel())[1:]) + 1
yy, xx = np.mgrid[0:mev.shape[0], 0:mev.shape[1]]
sobre_caixa = np.array([caixa[(xx - x) ** 2 + (yy - y) ** 2 <= r * r].mean() for x, y, r in c_toda])
falso = sobre_caixa > 0.5          # mais da metade do disco sobre a caixa de texto
na_faixa = c_toda[:, 1] >= tarja   # centro nas linhas da tarja
print(f"na imagem inteira: {len(c_toda)} círculos; {na_faixa.sum()} com centro nas linhas da tarja: "
      f"{falso.sum()} falsos (sobre a caixa de texto) e {(na_faixa & ~falso).sum()} partículas reais")
c = circulos(foto)
print(f"sem a tarja: {len(c)} círculos")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.imshow(mev, cmap="gray")
for (x, y, r), f in zip(c_toda, falso):
    ax.add_patch(plt.Circle((x, y), r, fill=False, color="red" if f else "orange", lw=1))
ax.set_axis_off()
ax.set_title("vermelho: círculos falsos, sobre a caixa de texto da tarja")
plt.show()

Só **2** círculos são falsos: os que caem sobre a caixa de texto. Os outros 3 com centro
nas linhas da tarja são partículas de verdade: duas parcialmente cobertas pela caixa,
uma ao lado dela. Recortar a tarja tira os falsos, mas leva junto essas 3 partículas
(as cobertas não seriam medidas direito de qualquer jeito).

In [ ]:
d_hough = 2 * c[:, 2]
print("raios encontrados (px):", np.unique(c[:, 2]))
print(f"diâmetro médio pelo Hough: {d_hough.mean() * NM_POR_PX:.0f} nm")

Repare nos raios: andam em degraus de uns 2 px. É o `dp=2`: com o acumulador na
metade da resolução, o raio não tem como ser mais fino que isso. Cada círculo sozinho
é grosseiro; a **média** de cem deles é estável.

## 8.5 — As duas rotas, lado a lado

Para comparar de forma justa, as duas rotas têm de medir **as mesmas partículas**. O
Hough só procura raios entre 12 e 22 px; o watershed vê também algumas partículas fora
dessa faixa (as pequenas, de uns 350 nm, e uma maior).

In [ ]:
faixa = (d_ws >= 2 * HOUGH["minRadius"]) & (d_ws <= 2 * HOUGH["maxRadius"])
print("partículas do watershed fora da faixa do Hough (nm):",
      np.round(d_ws[~faixa] * NM_POR_PX).astype(int))

ws_nm = d_ws[faixa].mean() * NM_POR_PX
hough_nm = d_hough.mean() * NM_POR_PX
print(f"limiar e contar manchas: {d_manchas.mean() * NM_POR_PX:5.0f} nm")
print(f"watershed:               {ws_nm:5.0f} nm")
print(f"Hough:                   {hough_nm:5.0f} nm")
print(f"diferença: {hough_nm - ws_nm:.0f} nm = {(hough_nm - ws_nm) / NM_POR_PX:.1f} px de diâmetro")

**Um pixel.** As duas rotas concordam a menos de 1 px de diâmetro, e essa diferença
tem uma causa: a borda de uma partícula não é um degrau, é uma **rampa** de alguns
pixels. O limiar põe a borda no meio da rampa; o Hough, onde o gradiente é máximo.

## 8.6 — Para o relatório

- Como o diâmetro encontrado aqui se compara com o que vocês mediram no **ImageJ**?
- E com o do **DLS**? O DLS mede o raio **hidrodinâmico**, na suspensão: espere um
  valor maior. Por quê?

# Exercícios

1. **(Otsu)** Rode `otsu_a_mao` na micrografia de MEV **sem** suavizar. O limiar muda?
   E a contagem de manchas?
2. **(marcadores)** No watershed da seção 8.3, troque `min_distance=10` por 5 e por 20.
   O que acontece com o número de partículas? Olhe as bacias e explique.
3. **(Hough)** Rode o Hough com `dp=1`. Os raios continuam em degraus? O diâmetro
   médio muda?
4. **(solidez)** Faça o histograma da solidez das bacias. Onde fica o corte natural?
   Quais regiões ficam abaixo de 0,9? Mostre-as na imagem.
5. **(multi-Otsu)** Use `threshold_multiotsu` com 3 e com 5 classes na imagem BSE. Qual
   número de fases o histograma sustenta?
6. **(sua imagem)** Aplique a seção 8 a outra micrografia do seu grupo. Lembre de medir
   a barra de escala dela.